# CYMEK FORMATION-MUX-001 + TIE-ROLE FRONTIER — Kaggle T4 x2

## BEFORE RUNNING

Kaggle **Settings → Accelerator → GPU T4 x2** and **Internet → ON**.

The two T4s are independent matched-seed arm workers; there is no DDP. The campaign is exact-resume and may span multiple Kaggle sessions. Near the session wall it stops launching new arms, packages state, and continues from attached Kaggle Output on the next run.

Stage 1 is frozen **Science S5**: CS-MECH-002 + REP-FORM-003A on the 60,000-row training surface, with the preregistered development-only VOCAB-PRESSURE-001 diagnostic.

Stage 2 is the prospectively preregistered **TIE-ROLE-FRONTIER-001**: TIE-ROLE-001 runs a 2x2 forward-equivalent tied-gradient factorial (canonical, input-gradient x4, output-gradient x0.25, balanced x4/x0.25) and TIE-ROLE-XFER-001 tests canonical vs balanced under frozen production-BPE rendering matched by actual processed tokens. This adds 24 official arms.

After all S5 and frontier development results are frozen, the coordinator runs two development-only mechanism diagnostics: VOCAB-PRESSURE-001 and an input-vs-output tied-gradient decomposition. Only then may raw sealed rows be regenerated in memory. Raw sealed examples are never persisted.

Operator v12 hardens coordinator protocol rebinding and performs a second storage-capacity preflight from real frontier calibration checkpoints before any frontier official arm launches.

If either state reports `PARTIAL_SESSION`, save the Kaggle version/output, attach that output to the next run, and rerun this exact notebook.


In [ ]:
# CELL 1 — fetch and verify immutable frontier operator + frozen Science S5
import os
# Architecture: allocator + streaming must be fixed before ANY worker
# process starts (env inheritance). Set here so the operator subprocess
# and its workers inherit them; setting after torch import is too late.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ['PYTHONUNBUFFERED'] = '1'
import pathlib, shutil, subprocess, sys
REPO = pathlib.Path('/kaggle/working/An-Ra-the-new-AGI')
REMOTE = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
OPERATOR_COMMIT = '4ee05f6e386f15d34f9dfa7bd7f3300a496b9896'
OPERATOR_PATH = 'tools/formation_mux_001_kaggle_operator_v12.py'
OPERATOR_BLOB = 'e9e1f701b0d4edc509194da55fe1ba37ed62ef86'
SCIENCE_COMMIT_S5 = 'c15ad8beb409537db42d075684ea54847a074ebd'
if REPO.exists() and not (REPO / '.git').exists():
    shutil.rmtree(REPO)
if not REPO.exists():
    subprocess.run(['git', 'clone', REMOTE, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'clean', '-fdx'], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-f', '-q', OPERATOR_COMMIT], check=True)
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
assert head == OPERATOR_COMMIT, (head, OPERATOR_COMMIT)
blob = subprocess.run(['git', '-C', str(REPO), 'hash-object', OPERATOR_PATH], capture_output=True, text=True, check=True).stdout.strip()
assert blob == OPERATOR_BLOB, (blob, OPERATOR_BLOB)
subprocess.run(['git', '-C', str(REPO), 'cat-file', '-e', SCIENCE_COMMIT_S5 + '^{commit}'], check=True)
# Architecture: fail closed in seconds on wrong hardware (no torch CUDA
# init in this kernel — query nvidia-smi only so we hold no GPU context).
smi = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
names = [l.strip() for l in smi.stdout.strip().splitlines() if l.strip()] if smi.returncode == 0 else []
print('GPUs visible:', names)
if len(names) != 2 or not all('T4' in n for n in names):
    raise RuntimeError('OFFICIAL RUN BLOCKED: set Kaggle Settings → Accelerator → GPU T4 x2 (observed: ' + str(names) + ')')
try:
    import tokenizers
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers'], check=True)
    import tokenizers
disk = shutil.disk_usage('/kaggle/working')
print('PYTHON:', sys.version.split()[0], '| TOKENIZERS:', tokenizers.__version__)
print('KAGGLE WORKING GiB total/free:', round(disk.total / 1024**3, 2), '/', round(disk.free / 1024**3, 2))
print('OPERATOR VERIFIED:', OPERATOR_COMMIT)
print('OPERATOR BLOB VERIFIED:', OPERATOR_BLOB)
print('SCIENCE S5 AVAILABLE:', SCIENCE_COMMIT_S5)


In [ ]:
# CELL 2 — GPU train session (T4 x2): train + dev-finalize, NO sealed
# Architecture (quota stretcher): sealed scoring + packaging move to
# the free CPU cell below. This cell burns GPU-h only on training.
import pathlib, subprocess, sys
root = pathlib.Path('/kaggle/working/FORMATION_MUX_001')
code = subprocess.run([
    sys.executable, '-u', 'tools/formation_mux_001_kaggle_operator_v12.py',
    '--repo', '/kaggle/working/An-Ra-the-new-AGI',
    '--out', str(root),
    '--skip-sealed',
], cwd='/kaggle/working/An-Ra-the-new-AGI')
if code.returncode != 0:
    failure = root / 'GLOBAL_FAILURE_V10.json'
    if failure.exists(): print('GLOBAL_FAILURE:', failure.read_text())
    raise RuntimeError('FORMATION-MUX frontier failed closed with exit ' + str(code.returncode) + '; preserve Kaggle Output and result ZIP')


In [ ]:
# CELL 2b — OPTIONAL 30-min clip triage (same T4 x2 session, after Cell 2)
# Runs the engineering-only M0 probe (no science change, no sealed).
# Skip freely: it is advisory for the S6 decision, never blocking.
import pathlib, subprocess, sys
_probe_root = pathlib.Path('/kaggle/working/FORMATION_MUX_001')
_probe = subprocess.run([
    sys.executable, '-u', 'tools/formation_mux_001_kaggle_operator_v12.py',
    '--repo', '/kaggle/working/An-Ra-the-new-AGI',
    '--out', str(_probe_root),
    '--lr-probe-only',
], cwd='/kaggle/working/An-Ra-the-new-AGI')
print('PROBE RETURN CODE:', _probe.returncode)
if _probe.returncode == 0:
    import json as _json
    _r = _json.loads((_probe_root / 'LR_CLIP_PROBE.json').read_text())
    print('PROBE VERDICT:', _r['verdict'])
    print('PROBE RECOMMENDATION:', _r['recommendation'])
else:
    print('Probe failed closed; campaign state untouched. Continue to the finalize/status cells regardless.')


In [ ]:
# CELL 2c — CPU finalize session (accelerator NONE): zero GPU-h.
# Run in a SEPARATE Kaggle session with Accelerator=None and this
# notebook's Output attached. Dev-finalize + diagnostics + one-shot
# sealed scoring + architecture gate + packaging, identical custody.
# Fails closed unless GPU Cell 2 left ARMS_COMPLETE on disk.
import pathlib, subprocess, sys
_fin_root = pathlib.Path('/kaggle/working/FORMATION_MUX_001')
_fin = subprocess.run([
    sys.executable, '-u', 'tools/formation_mux_001_kaggle_operator_v12.py',
    '--repo', '/kaggle/working/An-Ra-the-new-AGI',
    '--out', str(_fin_root),
    '--finalize-only',
], cwd='/kaggle/working/An-Ra-the-new-AGI')
print('FINALIZE RETURN CODE:', _fin.returncode)
if _fin.returncode != 0:
    raise RuntimeError('Finalize failed closed with exit ' + str(_fin.returncode) + '; attach the GPU session Output and rerun this cell')


In [ ]:
# CELL 3 — status / checkpoints / results / architecture gate
import json, pathlib
root = pathlib.Path('/kaggle/working/FORMATION_MUX_001')
bundle = pathlib.Path('/kaggle/working/FORMATION_MUX_001_RESULTS.zip')
s5 = json.loads((root / 'CAMPAIGN_STATE.json').read_text()) if (root / 'CAMPAIGN_STATE.json').exists() else {}
frontier = json.loads((root / 'TIE_ROLE_FRONTIER_STATE.json').read_text()) if (root / 'TIE_ROLE_FRONTIER_STATE.json').exists() else {}
print('S5 STATUS:', s5.get('status'), '| arms=', s5.get('complete_arms'), '/', s5.get('required_arms'))
print('FRONTIER STATUS:', frontier.get('status', 'not started'), '| arms=', frontier.get('complete_arms'), '/', frontier.get('required_arms'))
print('S5 WALL GUARD:', s5.get('wall_guard_triggered'), '| FRONTIER WALL GUARD:', frontier.get('wall_guard_triggered'))
q = root / 'QUALIFICATION.json'
if q.exists(): print('S5 QUALIFICATION:', json.loads(q.read_text()).get('status'))
fq = root / 'TIE_ROLE_FRONTIER_QUALIFICATION.json'
if fq.exists(): print('FRONTIER QUALIFICATION:', json.loads(fq.read_text()).get('status'))
fs = root / 'TIE_ROLE_STORAGE_PREFLIGHT.json'
if fs.exists():
    sr = json.loads(fs.read_text())
    print('FRONTIER STORAGE:', sr.get('pass'), '| free GiB=', round(sr.get('disk_free_bytes', 0)/1024**3, 2), '| required GiB=', round(sr.get('required_free_bytes', 0)/1024**3, 2))
x = root / 'CS-MECH-002' / 'VOCAB_PRESSURE_DIAGNOSTIC.json'
if x.exists():
    xr = json.loads(x.read_text()); focus = xr.get('focus_interpretation', {})
    print('VOCAB-PRESSURE:', focus.get('classification'), '| M2 rescue=', focus.get('mean_counterfactual_rescue_exact_valid_eos'))
g = root / 'TIE_ROLE_GRADIENT_DIAGNOSTIC.json'
if g.exists():
    gr = json.loads(g.read_text()); base = gr.get('per_arm', {}).get('TIE-ROLE-001/T0_CANONICAL', {})
    print('TIE-GRAD BASELINE: out/in=', base.get('mean_output_to_input_gradient_norm_ratio'), '| cosine=', base.get('mean_input_output_gradient_cosine'))
gate = root / 'TIE_ROLE_ARCHITECTURE_GATE.json'
if gate.exists(): print('ARCHITECTURE GATE:', json.loads(gate.read_text()))
for exp in ('CS-MECH-002', 'REP-FORM-003A', 'TIE-ROLE-001', 'TIE-ROLE-XFER-001'):
    p = root / exp / 'FINAL_RESULT.json'
    print(exp, '->', 'finalized' if p.exists() else 'not finalized')
print('RESULT ZIP:', bundle, 'exists=', bundle.exists())
if s5.get('status') == 'PARTIAL_SESSION' or frontier.get('status') == 'PARTIAL_SESSION':
    print('NEXT: save this Kaggle version/output, attach it to the next run, and rerun this exact notebook.')
